In [1]:
p = 0xffffffff00000001000000000000000000000000ffffffffffffffffffffffff
a = p - 3
E1 = EllipticCurve(GF(p),[a,3])
E2 = EllipticCurve(GF(p),[a,6])
E3 = EllipticCurve(GF(p),[a,18])
E4 = EllipticCurve(GF(p),[a,1])
E5 = EllipticCurve(GF(p),[a,4])
E6 = EllipticCurve(GF(p),[a,10])
E7 = EllipticCurve(GF(p),[a,11])
E8 = EllipticCurve(GF(p),[a,12])
E9 = EllipticCurve(GF(p),[a,13])
print(E9.order(), E9.cardinality())
print(E9.b_invariants())

115792089210356248762697446949407573529933855669854715991484147509429903655869 115792089210356248762697446949407573529933855669854715991484147509429903655869
(0, 115792089210356248762697446949407573530086143415290314195533631308867097853945, 52, 115792089210356248762697446949407573530086143415290314195533631308867097853942)


In [155]:
P_test = E1((86861948779320723735809409166338661013924870073533153401405380363591285609445,
            92959093380773739239256843672796491295827279525897738876018216604059003433908))
print(P_test[0],P_test[1],sep=',')
P = E1.random_point()
(P*7).order()

86861948779320723735809409166338661013924870073533153401405380363591285609445,92959093380773739239256843672796491295827279525897738876018216604059003433908


2067715878756361585048168695525135241607060617508089236978219051814511050518

In [3]:
def find_point_of_order(curve, n):
    psi3 = E1.division_polynomial(n)
    possible_x = psi3.roots(multiplicities=False)
    points_of_order_n = []
    for x in possible_x:
        try:
            P = curve.lift_x(x)  # поднимаем x до точки на кривой
            if P.order() == n:
                points_of_order_n.append(P)
        except:
            continue
    return points_of_order_n

find_point_of_order(E1,3)

[(89995002874197087156160429731648695860910221822426040658975619972952380673767 : 14349743460558275675129535079038365302062743301233311020938192075259646708873 : 1)]

In [4]:
def points_of_order(E, n):

    if n not in ZZ or n <= 0:
        raise ValueError("Порядок n должен быть положительным целым числом")
    
    # Получаем группу точек кривой над полем определения
    K = E.base_field()
    
    # Если поле конечное, используем эффективные методы
    if K.is_finite():
        N = E.cardinality()  # Порядок группы
        
        # Проверяем, может ли существовать точка порядка n
        if N % n != 0:
            return []
        
        # Вычисляем кофактор
        h = N // n
        
        points = []
        found_points = set()
        max_attempts = 10 * n  # Эвристика для количества попыток
        
        for _ in range(max_attempts):
            # Берем случайную точку
            try:
                x = K.random_element()
                P = E.lift_x(x)
            except:
                continue  # Если x не соответствует точке
            
            # Умножаем на кофактор
            Q = h * P
            
            # Проверяем, что получили точку нужного порядка
            if Q != E(0) and Q not in found_points:
                if Q.order() == n:
                    points.append(Q)
                    found_points.add(Q)
                    return points
                    # Если нам нужны все точки, можно продолжить поиск
                    # Но для больших групп это может быть долго
        
        return points
    
    # Для бесконечного поля (например, Q) ищем только точки кручения
    else:
        # Получаем все точки кручения
        torsion_points = E.torsion_points()
        
        # Фильтруем по порядку
        return [P for P in torsion_points if P != E(0) and P.order() == n]
    
points_of_order(E1,16033)

[]

In [58]:
temp_for_find_points = {E1 : [3,7,13,37,97,113],
                       E2 : [5,4003,16033],
                       E3 : [29,349,13998374731216357]}
result = []
for k, v in temp_for_find_points.items():
    for i in v:
        if i == 2:
            temp = find_point_of_order(k,i)
        else:
            temp = points_of_order(k, i)
        
        #набор значений для атаки в виде (b, l, 𝑅𝑖𝑛𝑣)
        result.append((k.a6(),i,(temp[0][0],temp[0][1]) if temp else []))

In [13]:
E_defolt = EllipticCurve(GF(p),[a,0x5ac635d8aa3a93e7b3ebbd55769886bc651d06b0cc53b0f63bce3c3e27d2604b])
A = E_defolt((337035179862752446972784639085065431250847102024630281174630997213779505939, 52482353919767272902722500030169999964387469385033390898927746707772624337801))
P = E_defolt((0x6b17d1f2e12c4247f8bce6e563a440f277037d812deb33a0f4a13945d898c296,
         0x4fe342e2fe1a7f9b8ee7eb4a7c0f9e162bce33576b315ececbb6406837bf51f5))
P * 181880515828151775623529011735461115006532

(50076794564799665691882082706870919195774533504164717129645073174459645245784 : 1976714381218771041551464137093264694976804623192229431706354209751649564437 : 1)

In [8]:
primes = prime_range(5, 800)
curves = [E1,E2,E3,E4,E5,E6,E7,E8,E9]
result = []
for i in primes:
    for j in curves_new:
        temp = points_of_order(j, i)
        if not temp:
            continue
        #набор значений для атаки в виде (b, l, 𝑅𝑖𝑛𝑣)
        result.append((j.a6(),i,(temp[0][0],temp[0][1]) if temp else []))
        break

(0 : 1 : 0)

In [7]:
curves_new =[EllipticCurve(GF(p),[a,33]), EllipticCurve(GF(p),[a,34])
,EllipticCurve(GF(p),[a,35])
,EllipticCurve(GF(p),[a,41])
,EllipticCurve(GF(p),[a,42])]